# Choquet regression workflow

Choquet regression replaces an additive linear predictor by `y = beta_0 + C_nu(x) + error`. The workflow follows the thesis: orient and normalize predictors, choose capacity complexity, fit under capacity constraints, evaluate, and interpret.


In [15]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.pipeline import Pipeline

from capacities_ml_fin.base.interpretation import pairwise_interaction_matrix, pairwise_interactions, shapley_indices
from capacities_ml_fin.ml.model_selection import capacity_parameter_grid
from capacities_ml_fin.ml.models import ChoquetRegressor
from capacities_ml_fin.ml.optimization import L1Penalty, Solver
from capacities_ml_fin.ml.preprocessing import CapacityNormalizer

rng = np.random.default_rng(11)


## 1. Data and train/test split

`volatility` is a cost criterion: after preprocessing, larger values always mean a more desirable outcome.


In [16]:
n = 90
X = pd.DataFrame(
    {
        "profitability": rng.uniform(5.0, 25.0, n),
        "liquidity": rng.uniform(0.8, 2.5, n),
        "volatility": rng.uniform(0.10, 0.60, n),
    }
)
oriented = np.column_stack(
    (
        (X["profitability"] - 5.0) / 20.0,
        (X["liquidity"] - 0.8) / 1.7,
        (0.60 - X["volatility"]) / 0.50,
    )
)
y = (
    0.20
    + 0.25 * oriented[:, 0]
    + 0.20 * oriented[:, 1]
    + 0.15 * oriented[:, 2]
    + 0.25 * np.minimum(oriented[:, 0], oriented[:, 1])
    + 0.15 * np.minimum(oriented[:, 1], oriented[:, 2])
    + rng.normal(0.0, 0.025, n)
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=7
)
display(X.head())
print("First y values:", y[:5])


,profitability,liquidity,volatility
0,7.571404,1.216660,0.270041
1,14.985557,2.236882,0.474922
2,17.029967,2.061072,0.292397
3,5.573780,1.727850,0.176622
4,7.958522,1.924508,0.538228


First y values: [0.45828032 0.67964558 0.8722495  0.50114172 0.45590679]


## 2. Preprocessing and capacity selection

The normalizer learns ranges from each training fold. Cross-validation compares additive, 2-additive, and full 3-additive Möbius representations.


In [17]:
pipeline = Pipeline(
    [
        ("normalize", CapacityNormalizer(cost_features=["volatility"])),
        (
            "model",
            ChoquetRegressor(
                solver=Solver.SCIPY,
                solver_options={"options": {"maxiter": 1500, "ftol": 1e-10}},
            ),
        ),
    ]
).set_output(transform="pandas")

search = GridSearchCV(
    pipeline,
    capacity_parameter_grid(parameter_name="model__sparsity", orders=(1, 2, 3)),
    scoring="neg_mean_squared_error",
    cv=KFold(n_splits=3, shuffle=True, random_state=4),
    n_jobs=1,
)
search.fit(X_train, y_train)

selection_results = pd.DataFrame(search.cv_results_)[
    ["param_model__sparsity", "mean_test_score", "std_test_score"]
]
selection_results["mean_validation_mse"] = -selection_results["mean_test_score"]
selection_results["mean_validation_rmse"] = np.sqrt(selection_results["mean_validation_mse"])
display(
    selection_results[
        ["param_model__sparsity", "mean_validation_mse", "mean_validation_rmse"]
    ]
)
print("Selected capacity:", search.best_params_["model__sparsity"])


,param_model__sparsity,mean_validation_mse,mean_validation_rmse
0,"KAdditivity(order=1, shape=<CapacityShape.GENE...",0.001623,0.040288
1,"KAdditivity(order=2, shape=<CapacityShape.GENE...",0.000783,0.027981
2,"KAdditivity(order=3, shape=<CapacityShape.GENE...",0.000667,0.025822


Selected capacity: KAdditivity(order=3, shape=<CapacityShape.GENERAL: 'general'>)


## 3. Interaction regularization and final fit

Following the thesis, L1 regularization is applied only to Möbius coefficients with order at least two. Singleton effects are not penalized.


In [18]:
best_sparsity = search.best_params_["model__sparsity"]
compilation = best_sparsity.compile(X.shape[1])
interaction_positions = np.array(
    [
        position
        for position, mask in enumerate(compilation.bundle.parameter_masks)
        if mask.bit_count() >= 2
    ],
    dtype=int,
)
penalty = L1Penalty(weight=5e-4, selection=interaction_positions)

final_pipeline = Pipeline(
    [
        ("normalize", CapacityNormalizer(cost_features=["volatility"])),
        (
            "model",
            ChoquetRegressor(
                sparsity=best_sparsity,
                penalty=penalty,
                solver=Solver.SCIPY,
                solver_options={"options": {"maxiter": 2000, "ftol": 1e-12}},
            ),
        ),
    ]
).set_output(transform="pandas").fit(X_train, y_train)


## 4. Out-of-sample evaluation

MSE, RMSE, and MAE are better when lower; R² is better when closer to one. A classical linear regression receives the same normalized inputs, so the comparison isolates the nonlinear interactions introduced by the Choquet integral. The mean predictor remains as a minimal baseline.


In [19]:
classical_pipeline = Pipeline(
    [
        ("normalize", CapacityNormalizer(cost_features=["volatility"])),
        ("model", LinearRegression()),
    ]
).set_output(transform="pandas").fit(X_train, y_train)

train_prediction = final_pipeline.predict(X_train)
test_prediction = final_pipeline.predict(X_test)
linear_prediction = classical_pipeline.predict(X_test)
baseline_prediction = np.full(y_test.shape, np.mean(y_train), dtype=float)

def regression_metrics(observed, predicted):
    mse = mean_squared_error(observed, predicted)
    return {
        "MSE": mse,
        "RMSE": mse ** 0.5,
        "MAE": mean_absolute_error(observed, predicted),
        "R2": r2_score(observed, predicted),
    }

evaluation = pd.DataFrame(
    {
        "train": regression_metrics(y_train, train_prediction),
        "Choquet test": regression_metrics(y_test, test_prediction),
        "linear regression": regression_metrics(y_test, linear_prediction),
        "mean baseline": regression_metrics(y_test, baseline_prediction),
    }
)
display(evaluation)
display(pd.DataFrame({"observed": y_test[:8], "predicted": test_prediction[:8]}))


,train,Choquet test,linear regression,mean baseline
MSE,0.000546,0.001288,0.003031,0.037632
RMSE,0.023357,0.035894,0.055050,0.193989
MAE,0.018495,0.025273,0.041405,0.168996
R2,0.978005,0.965188,0.918115,-0.016801


,observed,predicted
0,0.554990,0.585769
1,0.713651,0.714104
2,1.087388,1.079146
3,0.715550,0.701735
4,0.774588,0.817434
5,0.439413,0.432511
6,0.723551,0.746293
7,0.666437,0.681386


## 5. Interpret the fitted capacity


In [20]:
fitted_model = final_pipeline.named_steps["model"]
display(pd.Series(shapley_indices(fitted_model.capacity_), name="Shapley importance"))
display(pd.Series(pairwise_interactions(fitted_model.capacity_), name="pairwise interaction"))
display(
    pd.DataFrame(
        pairwise_interaction_matrix(fitted_model.capacity_),
        index=X.columns,
        columns=X.columns,
    )
)


profitability    0.372054
liquidity        0.380111
volatility       0.247835
Name: Shapley importance, dtype: float64

profitability  liquidity     0.197394
               volatility    0.034256
liquidity      volatility    0.112900
Name: pairwise interaction, dtype: float64

,profitability,liquidity,volatility
profitability,0.000000,0.197394,0.034256
liquidity,0.197394,0.000000,0.112900
volatility,0.034256,0.112900,0.000000


## 6. Predict new observations


In [21]:
X_new = pd.DataFrame(
    {
        "profitability": [10.0, 18.0, 23.0],
        "liquidity": [1.0, 1.8, 2.2],
        "volatility": [0.50, 0.30, 0.15],
    }
)
display(X_new.assign(prediction=final_pipeline.predict(X_new)))


,profitability,liquidity,volatility,prediction
0,10.0,1.0,0.50,0.347913
1,18.0,1.8,0.30,0.803639
2,23.0,2.2,0.15,1.064121
